<a href="https://colab.research.google.com/github/ESHANBUDHRANI/BehavioralEdgeAI-2.0/blob/main/Behavioral_Trading_Colab_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧠 Explainable Risk Attribution with Agentic System
**Behavioral Trading Analysis** — Upload trades → Enrich with market data → Run 12 models → Visualise → Chat with Groq LLM

> Run cells **top to bottom** on first use. After setup, individual sections can be re-run independently.

## Section 1 — Setup
### Cell 1: Install Packages

In [1]:
!pip install -q pandas numpy scipy scikit-learn yfinance ta \
    transformers torch hmmlearn pgmpy statsmodels arch shap plotly \
    chromadb langchain langchain-community langchain-groq \
    langchain-huggingface sentence-transformers copulas ruptures groq
print("✅ Packages installed.")


  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.0/166.0 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.3/981.3 kB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 96.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 71.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 87.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 103.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━

### Cell 2: Drive Mount, GPU Check & Imports

In [3]:
from google.colab import drive
import torch, os, uuid, json, sqlite3, io, warnings
warnings.filterwarnings('ignore')

drive.mount('/content/drive')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"✅ Device: {device}")

base_path = '/content/behavioral_analysis'
os.makedirs(base_path, exist_ok=True)
os.makedirs(f'{base_path}/chroma_db', exist_ok=True)
os.makedirs(f'{base_path}/cache', exist_ok=True)

import pandas as pd
import numpy as np
from scipy.stats import gaussian_kde, skew, kurtosis
from scipy.optimize import curve_fit
from sklearn.mixture import GaussianMixture
from sklearn.cluster import AgglomerativeClustering
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from hmmlearn import hmm
from arch import arch_model
from statsmodels.tsa.stattools import grangercausalitytests
import torch.nn as nn, torch.optim as optim
import shap
from copulas.multivariate import GaussianMultivariate
import yfinance as yf
import ta as ta_lib
import chromadb
from chromadb.utils import embedding_functions
from collections import deque
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display, HTML

session_id = str(uuid.uuid4())
print(f"✅ Session ID: {session_id}")
print("✅ All imports successful.")


Mounted at /content/drive
✅ Device: cuda
✅ Session ID: e2d4f1b5-9b7d-450d-881c-14dd0b065f0b
✅ All imports successful.


### Cell 3: SQLite Schema

In [4]:
db_path = f'{base_path}/trades.db'
db_conn = sqlite3.connect(db_path)

def create_tables(conn):
    conn.executescript("""
    CREATE TABLE IF NOT EXISTS trades (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        session_id TEXT, timestamp TEXT, symbol TEXT,
        side TEXT, quantity REAL, price REAL,
        pnl REAL, holding_duration REAL, emergency INTEGER DEFAULT 0
    );
    CREATE TABLE IF NOT EXISTS market_context (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        session_id TEXT, symbol TEXT, date TEXT,
        open REAL, high REAL, low REAL, close REAL, volume REAL,
        sma20 REAL, sma50 REAL, ema20 REAL, ema50 REAL,
        rsi REAL, macd REAL, macd_signal REAL, atr REAL,
        bb_upper REAL, bb_lower REAL, adx REAL,
        obv REAL, volume_z REAL, sentiment_score REAL,
        sentiment_label TEXT, market_regime TEXT
    );
    """)
    conn.commit()

create_tables(db_conn)
print("✅ SQLite schema ready.")


✅ SQLite schema ready.


### Cell 4: API Keys

In [7]:
from google.colab import userdata

# Add your Groq key: Runtime > Secrets (🔑) > GROQ_API_KEY = gsk_xxxx
# Free key at: https://console.groq.com/keys
try:
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
    print("✅ Groq API key loaded from Colab Secrets.")
except Exception:
    GROQ_API_KEY = ""
    print("⚠️  GROQ_API_KEY not in Secrets. Add it via Runtime > Secrets.")


✅ Groq API key loaded from Colab Secrets.


## Section 2 — Data Ingestion
### Cell 5: Upload CSV

In [8]:
from google.colab import files

print("📂 Upload your trade history CSV.")
print("Required columns: date/timestamp, symbol, buy/sell side, quantity, price")
print("Optional: pnl, holding_duration\n")

uploaded = files.upload()
filename = list(uploaded.keys())[0]
df_raw = pd.read_csv(io.BytesIO(uploaded[filename]))
print(f"✅ Loaded {len(df_raw)} rows, {len(df_raw.columns)} columns.")
display(df_raw.head())


📂 Upload your trade history CSV.
Required columns: date/timestamp, symbol, buy/sell side, quantity, price
Optional: pnl, holding_duration



Saving sample_trades.csv to sample_trades.csv
✅ Loaded 96 rows, 7 columns.


,timestamp,symbol,buy_sell,quantity,price,pnl,holding_duration
0,2024-01-03 09:30:00,RELIANCE.NS,BUY,10,2450.0,0.0,0.0
1,2024-01-05 14:15:00,RELIANCE.NS,SELL,10,2510.0,600.0,2.2
2,2024-01-08 10:00:00,TCS.NS,BUY,5,3800.0,0.0,0.0
3,2024-01-10 11:30:00,TCS.NS,SELL,5,3750.0,-250.0,2.1
4,2024-01-11 09:45:00,TCS.NS,BUY,8,3740.0,0.0,0.0


### Cell 6: Column Mapping & Normalisation

In [9]:
# Auto-detect columns — edit the map below if auto-detection is wrong
AUTO_MAP = {
    'timestamp': ['timestamp','date','datetime','time','trade_date','date/time'],
    'symbol':    ['symbol','ticker','instrument','stock','scrip'],
    'side':      ['side','buy_sell','type','direction','action','buy/sell'],
    'quantity':  ['quantity','qty','shares','units','volume','lot'],
    'price':     ['price','avg_price','fill_price','trade_price','rate'],
}

def auto_detect(df, auto_map):
    mapping = {}
    for canonical, aliases in auto_map.items():
        for col in df.columns:
            if col.strip().lower().replace(' ','_') in aliases:
                mapping[col] = canonical
                break
    return mapping

detected = auto_detect(df_raw, AUTO_MAP)
print("Auto-detected column mapping:", detected)
print("If wrong, manually set: detected['your_col'] = 'canonical_name'")

df = df_raw.rename(columns=detected).copy()

# Ensure required columns exist
required = ['timestamp','symbol','side','quantity','price']
missing = [r for r in required if r not in df.columns]
if missing:
    print(f"⚠️  Missing columns: {missing}")
    print("Manually map them: df = df.rename(columns={'your_col': 'timestamp'}) etc.")
else:
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
    df['symbol']    = df['symbol'].astype(str).str.upper().str.strip()
    df['side']      = df['side'].astype(str).str.upper().str.strip()
    df['quantity']  = pd.to_numeric(df['quantity'], errors='coerce')
    df['price']     = pd.to_numeric(df['price'], errors='coerce')
    df = df.dropna(subset=['timestamp','symbol','side','quantity','price'])
    df = df.drop_duplicates(subset=['timestamp','symbol','side','quantity','price'])
    df = df.sort_values('timestamp').reset_index(drop=True)
    print(f"✅ Normalised: {len(df)} trades across {df['symbol'].nunique()} symbols.")
    print(f"   Date range: {df['timestamp'].min().date()} → {df['timestamp'].max().date()}")
    display(df.head())


Auto-detected column mapping: {'timestamp': 'timestamp', 'symbol': 'symbol', 'buy_sell': 'side', 'quantity': 'quantity', 'price': 'price'}
If wrong, manually set: detected['your_col'] = 'canonical_name'
✅ Normalised: 96 trades across 6 symbols.
   Date range: 2024-01-03 → 2024-07-12


,timestamp,symbol,side,quantity,price,pnl,holding_duration
0,2024-01-03 09:30:00,RELIANCE.NS,BUY,10,2450.0,0.0,0.0
1,2024-01-05 14:15:00,RELIANCE.NS,SELL,10,2510.0,600.0,2.2
2,2024-01-08 10:00:00,TCS.NS,BUY,5,3800.0,0.0,0.0
3,2024-01-10 11:30:00,TCS.NS,SELL,5,3750.0,-250.0,2.1
4,2024-01-11 09:45:00,TCS.NS,BUY,8,3740.0,0.0,0.0


### Cell 7: FIFO Position Reconstruction & DB Persistence

In [10]:
def reconstruct_fifo(df, sid, conn):
    inventory = {}
    realized = []
    for _, row in df.sort_values('timestamp').iterrows():
        sym = row['symbol']
        if sym not in inventory: inventory[sym] = deque()
        if row['side'] == 'BUY':
            inventory[sym].append({'price': row['price'], 'qty': row['quantity'], 'time': row['timestamp']})
        elif row['side'] == 'SELL':
            sell_qty, wpnl, whold, matched = row['quantity'], 0.0, 0.0, 0.0
            while sell_qty > 0 and inventory.get(sym):
                lot = inventory[sym][0]
                mq = min(lot['qty'], sell_qty)
                lot['qty'] -= mq; sell_qty -= mq; matched += mq
                wpnl  += (row['price'] - lot['price']) * mq
                whold += (row['timestamp'] - lot['time']).total_seconds() / 86400 * mq
                if lot['qty'] <= 1e-9: inventory[sym].popleft()
            avg_hold = whold / matched if matched else 0.0
            realized.append({
                'session_id': sid, 'timestamp': str(row['timestamp']),
                'symbol': sym, 'side': 'SELL',
                'quantity': row['quantity'], 'price': row['price'],
                'pnl': wpnl, 'holding_duration': avg_hold, 'emergency': 0
            })
    if realized:
        pd.DataFrame(realized).to_sql('trades', conn, if_exists='append', index=False)
        conn.commit()
    return pd.DataFrame(realized)

df_realized = reconstruct_fifo(df, session_id, db_conn)
wins = (df_realized['pnl'] > 0).sum()
losses = (df_realized['pnl'] < 0).sum()
print(f"✅ FIFO reconstruction: {len(df_realized)} realized trades")
print(f"   Wins: {wins} | Losses: {losses} | Win rate: {wins/max(wins+losses,1):.1%}")
print(f"   Total PnL: {df_realized['pnl'].sum():.2f}")


✅ FIFO reconstruction: 48 realized trades
   Wins: 27 | Losses: 21 | Win rate: 56.2%
   Total PnL: 8780.00


### Cell 8: Emergency Trade Flagging

In [13]:
import ipywidgets as widgets
from IPython.display import display

print("Review your trades below. Identify any executed due to personal financial emergency.")
print("These will be EXCLUDED from behavioral modeling.\n")
display(df_realized[['timestamp','symbol','side','quantity','price','pnl']].head(30))

# Widget-based input instead of input()
emergency_text = widgets.Text(
    placeholder='Enter row numbers e.g. 2,5,8 — or leave blank for none',
    description='Emergency rows:',
    layout=widgets.Layout(width='500px')
)
confirm_btn = widgets.Button(description='Confirm & Continue', button_style='primary')
output = widgets.Output()

def on_confirm(b):
    with output:
        val = emergency_text.value.strip()
        global emergency_trade_ids, modeling_df
        emergency_trade_ids = [int(x.strip()) for x in val.split(',') if x.strip().isdigit()] if val else []
        if emergency_trade_ids:
            db_conn.executemany("UPDATE trades SET emergency = 1 WHERE rowid = ?",
                                [(i+1,) for i in emergency_trade_ids])
            db_conn.commit()
        modeling_df = df_realized[~df_realized.index.isin(emergency_trade_ids)].copy()
        print(f"✅ Emergency trades excluded: {len(emergency_trade_ids)}")
        print(f"   Trades for modeling: {len(modeling_df)}")

confirm_btn.on_click(on_confirm)
display(emergency_text, confirm_btn, output)

Review your trades below. Identify any executed due to personal financial emergency.
These will be EXCLUDED from behavioral modeling.



,timestamp,symbol,side,quantity,price,pnl
0,2024-01-05 14:15:00,RELIANCE.NS,SELL,10,2510.0,600.0
1,2024-01-10 11:30:00,TCS.NS,SELL,5,3750.0,-250.0
2,2024-01-12 15:00:00,TCS.NS,SELL,8,3690.0,-400.0
3,2024-01-18 13:00:00,INFY.NS,SELL,15,1680.0,900.0
4,2024-01-23 14:30:00,HDFCBANK.NS,SELL,12,1620.0,480.0
5,2024-01-26 11:00:00,RELIANCE.NS,SELL,20,2440.0,-800.0
6,2024-01-31 15:15:00,WIPRO.NS,SELL,25,495.0,375.0
7,2024-02-05 14:00:00,TCS.NS,SELL,6,3810.0,540.0
8,2024-02-08 11:45:00,INFY.NS,SELL,20,1630.0,-400.0
9,2024-02-09 15:30:00,INFY.NS,SELL,30,1610.0,-450.0


Text(value='', description='Emergency rows:', layout=Layout(width='500px'), placeholder='Enter row numbers e.g…

Button(button_style='primary', description='Confirm & Continue', style=ButtonStyle())

Output()

## Section 3 — Market Context
### Cell 9: Fetch Market Data with Indicators

In [16]:
def flatten_yf(df):
    """Fix yfinance MultiIndex columns and header rows."""
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [col[0] if isinstance(col, tuple) else col for col in df.columns]
    df.columns = [str(c).strip().title() for c in df.columns]
    if 'Close' in df.columns:
        df = df[pd.to_numeric(df['Close'], errors='coerce').notna()].copy()
        for c in df.columns:
            if c not in ['Date','Datetime']:
                df[c] = pd.to_numeric(df[c], errors='coerce')
    return df.reset_index(drop=True)

def compute_indicators(df):
    if df.empty: return df
    df = flatten_yf(df.reset_index())
    date_col = [c for c in df.columns if 'date' in c.lower() or 'datetime' in c.lower()]
    if date_col: df = df.rename(columns={date_col[0]: 'Date'})
    c, h, l, v = 'Close', 'High', 'Low', 'Volume'
    df['SMA20']          = ta_lib.trend.sma_indicator(df[c], 20)
    df['SMA50']          = ta_lib.trend.sma_indicator(df[c], 50)
    df['EMA20']          = ta_lib.trend.ema_indicator(df[c], 20)
    df['EMA50']          = ta_lib.trend.ema_indicator(df[c], 50)
    macd                 = ta_lib.trend.MACD(df[c])
    df['MACD']           = macd.macd()
    df['MACDH']          = macd.macd_diff()
    adx                  = ta_lib.trend.ADXIndicator(df[h], df[l], df[c])
    df['ADX_14']         = adx.adx()
    df['RSI']            = ta_lib.momentum.rsi(df[c], 14)
    df['ATR']            = ta_lib.volatility.average_true_range(df[h], df[l], df[c])
    df['ATR_Rolling_Mean'] = df['ATR'].rolling(20).mean()
    bb                   = ta_lib.volatility.BollingerBands(df[c])
    df['BB_upper']       = bb.bollinger_hband()
    df['BB_lower']       = bb.bollinger_lband()
    df['BB_bandwidth']   = bb.bollinger_wband()
    df['OBV']            = ta_lib.volume.on_balance_volume(df[c], df[v]) if v in df.columns else 0
    df['Volume_Z']       = (df[v] - df[v].rolling(20).mean()) / df[v].rolling(20).std() if v in df.columns else 0
    df['Volatility']     = df[c].pct_change().rolling(20).std()
    return df.replace([np.inf, -np.inf], np.nan)

def classify_regime(row):
    try:
        atr_m = row.get('ATR_Rolling_Mean', 0) or 0
        if row.get('ATR', 0) > 1.5 * atr_m: return 'high_volatility'
        if atr_m and row.get('ATR', 0) < 0.7 * atr_m: return 'low_volatility'
        if (row.get('ADX_14', 0) or 0) > 25:
            if row.get('EMA20', 0) > row.get('EMA50', 0): return 'trending_bullish'
            if row.get('EMA20', 0) < row.get('EMA50', 0): return 'trending_bearish'
        if (row.get('RSI', 50) or 50) < 35: return 'risk_off'
        if (row.get('ADX_14', 25) or 25) < 20: return 'ranging'
        return 'normal'
    except: return 'normal'

# FIX: convert timestamp to datetime before using it
modeling_df['timestamp'] = pd.to_datetime(modeling_df['timestamp'], errors='coerce')

symbols = modeling_df['symbol'].unique().tolist()
start = (modeling_df['timestamp'].min() - pd.Timedelta(days=260)).strftime('%Y-%m-%d')
end   = (modeling_df['timestamp'].max() + pd.Timedelta(days=5)).strftime('%Y-%m-%d')
market_data = {}

print(f"Fetching daily OHLCV for {len(symbols)} symbols ({start} → {end})...")
for sym in symbols:
    cache_file = f'{base_path}/cache/{sym}.csv'
    try:
        if os.path.exists(cache_file):
            raw = pd.read_csv(cache_file)
        else:
            raw = yf.download(sym, start=start, end=end, interval='1d', progress=False, auto_adjust=False)
            raw.to_csv(cache_file)
        raw = compute_indicators(raw)
        raw = raw.set_index('Date').sort_index() if 'Date' in raw.columns else raw
        raw.index = pd.to_datetime(raw.index, errors='coerce')
        if 'RSI' in raw.columns:
            rsi_s  = (raw['RSI'] - 50) / 50
            macd_s = (raw['MACDH'] / (raw['Close'].rolling(20).std() + 1e-9)).clip(-1, 1)
            mom    = raw['Close'].pct_change(10).fillna(0)
            raw['sentiment_score'] = (0.30*rsi_s + 0.25*macd_s + 0.20*mom).fillna(0)
            raw['sentiment_label'] = np.select(
                [raw['sentiment_score'] > 0.2, raw['sentiment_score'] < -0.2],
                ['bullish','bearish'], default='neutral')
        raw['market_regime'] = raw.apply(classify_regime, axis=1)
        market_data[sym] = raw
        print(f"  ✅ {sym}: {len(raw)} trading days")
    except Exception as e:
        print(f"  ❌ {sym}: {e}")

print(f"\n✅ Market data ready for {len(market_data)}/{len(symbols)} symbols.")

Fetching daily OHLCV for 6 symbols (2023-04-20 → 2024-07-17)...
  ✅ RELIANCE.NS: 305 trading days
  ✅ TCS.NS: 305 trading days
  ✅ INFY.NS: 305 trading days
  ✅ HDFCBANK.NS: 305 trading days
  ✅ WIPRO.NS: 305 trading days
  ✅ ICICIBANK.NS: 305 trading days

✅ Market data ready for 6/6 symbols.


### Cell 10: Enrich Trades with Market Context

In [17]:
enriched_list = []
for _, trade in modeling_df.iterrows():
    sym = trade['symbol']
    if sym not in market_data: continue
    mdf = market_data[sym]
    ts  = pd.Timestamp(trade['timestamp']).tz_localize(None)
    available = mdf[mdf.index <= ts]
    if available.empty: continue
    ctx = available.iloc[-1].to_dict()
    enriched_list.append({**trade.to_dict(), **ctx})

enriched_trades = pd.DataFrame(enriched_list)
print(f"✅ Enriched {len(enriched_trades)} trades with market context.")

# Persist to DB
ctx_rows = []
for sym, data in market_data.items():
    for date, row in data.iterrows():
        ctx_rows.append({
            'session_id': session_id, 'symbol': sym,
            'date': str(date.date()), 'open': row.get('Open'), 'high': row.get('High'),
            'low': row.get('Low'), 'close': row.get('Close'), 'volume': row.get('Volume'),
            'sma20': row.get('SMA20'), 'sma50': row.get('SMA50'),
            'ema20': row.get('EMA20'), 'ema50': row.get('EMA50'),
            'rsi': row.get('RSI'), 'macd': row.get('MACD'),
            'macd_signal': None, 'atr': row.get('ATR'),
            'bb_upper': row.get('BB_upper'), 'bb_lower': row.get('BB_lower'),
            'adx': row.get('ADX_14'), 'obv': row.get('OBV'),
            'volume_z': row.get('Volume_Z'),
            'sentiment_score': row.get('sentiment_score', 0),
            'sentiment_label': row.get('sentiment_label', 'neutral'),
            'market_regime': row.get('market_regime', 'normal')
        })
pd.DataFrame(ctx_rows).to_sql('market_context', db_conn, if_exists='append', index=False)
db_conn.commit()
display(enriched_trades[['timestamp','symbol','side','pnl','RSI','ADX_14','market_regime']].head())


✅ Enriched 44 trades with market context.


,timestamp,symbol,side,pnl,RSI,ADX_14,market_regime
0,2024-01-05 14:15:00,RELIANCE.NS,SELL,600.0,68.891800,44.298560,trending_bullish
1,2024-01-10 11:30:00,TCS.NS,SELL,-250.0,51.819630,28.105596,trending_bullish
2,2024-01-18 13:00:00,INFY.NS,SELL,900.0,65.455499,22.047098,normal
3,2024-01-26 11:00:00,RELIANCE.NS,SELL,-800.0,60.273118,44.116262,trending_bullish
4,2024-02-05 14:00:00,TCS.NS,SELL,540.0,66.213334,25.266112,trending_bullish


## Section 4 — Behavioral Feature Engineering
### Cell 11: Full Feature Set

In [18]:
def compute_all_features(df):
    df = df.copy().sort_values('timestamp').reset_index(drop=True)
    df['position_value'] = df['quantity'] * df['price']

    # Position size deviation from 20-trade rolling mean
    pv_mean = df['position_value'].rolling(20, min_periods=1).mean()
    pv_std  = df['position_value'].rolling(20, min_periods=1).std().replace(0, np.nan)
    df['size_deviation'] = (df['position_value'] - pv_mean) / pv_std

    df['is_loss'] = (df['pnl'] < 0).astype(int)

    # Post-loss time gap
    df['last_loss_time'] = df['timestamp'].where(df['is_loss'] == 1).ffill().shift(1)
    df['post_loss_hours'] = (df['timestamp'] - df['last_loss_time']).dt.total_seconds() / 3600

    # Trade frequency 7-day rolling
    df = df.set_index('timestamp')
    df['trade_freq_7d'] = df.assign(one=1)['one'].rolling('7D').sum()
    df = df.reset_index()
    freq_mean = df['trade_freq_7d'].mean()
    freq_std  = df['trade_freq_7d'].std() or 1
    df['frequency_spike'] = (df['trade_freq_7d'] > freq_mean + 2*freq_std).astype(int)

    # Early exit
    avg_hold = df['holding_duration'].mean() or 1
    df['early_exit'] = (df['holding_duration'] < 0.2 * avg_hold).astype(int)

    # RSI alignment (trade profitable when RSI agreed)
    if 'RSI' in df.columns:
        df['rsi_aligned'] = ((df['pnl'] > 0) & (df['RSI'] > 50) |
                             (df['pnl'] < 0) & (df['RSI'] < 50)).astype(int)
    else:
        df['rsi_aligned'] = 0

    # Revenge score
    plh   = df['post_loss_hours'].fillna(df['post_loss_hours'].median())
    plh_n = ((24 - plh.clip(0,24)) / 24).clip(0,1)
    sz_n  = df['size_deviation'].fillna(0).abs().clip(0,2) / 2
    df['revenge_score'] = (plh_n + sz_n + df['frequency_spike']) / 3.0

    # Emotional score & state
    df['emotional_score'] = (
        df['revenge_score'] * 0.4
        + df['size_deviation'].fillna(0).abs().clip(0,2) / 2 * 0.3
        + df['early_exit'] * 0.3
    )
    df['emotional_state'] = np.select(
        [df['emotional_score'] < 0.33, df['emotional_score'] < 0.66],
        ['calm', 'anxious'], default='euphoric'
    )
    return df.fillna(0)

enriched_trades = compute_all_features(enriched_trades)
print(f"✅ {enriched_trades.shape[1]} features computed for {len(enriched_trades)} trades.")
print(f"   Emotional distribution: {enriched_trades['emotional_state'].value_counts().to_dict()}")


✅ 46 features computed for 44 trades.
   Emotional distribution: {'calm': 39, 'anxious': 4, 'euphoric': 1}


### Cell 12: Baseline Statistics

In [19]:
try:
    if 'market_regime' in enriched_trades.columns:
        stats = enriched_trades.groupby('market_regime')['emotional_score'].agg(
            ['mean','std','median','count']).reset_index()
        print("Emotional score by market regime:")
        display(stats)
    win_rate = (enriched_trades['pnl'] > 0).mean()
    print(f"\nWin rate: {win_rate:.1%}")
    print(f"Avg holding duration: {enriched_trades['holding_duration'].mean():.1f} days")
    print(f"Avg position size: {enriched_trades['position_value'].mean():.2f}")
except Exception as e: print(f"Baseline error: {e}")


Emotional score by market regime:


,market_regime,mean,std,median,count
0,normal,0.157343,0.147163,0.136754,10
1,ranging,0.160353,0.137670,0.124758,13
2,risk_off,0.071837,0.078015,0.071837,2
3,trending_bearish,0.253745,0.241354,0.170309,6
4,trending_bullish,0.176437,0.156691,0.153206,13



Win rate: 56.8%
Avg holding duration: 2.6 days
Avg position size: 30142.27


## Section 5 — ML Models
### Cell 13: Feature Matrix Preparation

In [20]:
FEATURE_COLS = [
    'position_value', 'size_deviation', 'revenge_score',
    'emotional_score', 'post_loss_hours', 'trade_freq_7d',
    'frequency_spike', 'early_exit', 'holding_duration', 'pnl'
]
if 'RSI' in enriched_trades.columns:     FEATURE_COLS += ['RSI','ADX_14','ATR','sentiment_score']
if 'Volatility' in enriched_trades.columns: FEATURE_COLS += ['Volatility']

feat_cols = [c for c in FEATURE_COLS if c in enriched_trades.columns]
X_raw = enriched_trades[feat_cols].fillna(0).replace([np.inf, -np.inf], 0).to_numpy()
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)
print(f"✅ Feature matrix: {X.shape[0]} trades × {X.shape[1]} features")


✅ Feature matrix: 44 trades × 15 features


### Cell 14: GMM Clustering

In [21]:
# GMM
gmm = GaussianMixture(n_components=4, covariance_type='full', random_state=42)
enriched_trades['gmm_cluster'] = gmm.fit_predict(X)

# Agglomerative
agg = AgglomerativeClustering(n_clusters=4)
enriched_trades['agg_cluster'] = agg.fit_predict(X)

gmm_sil = silhouette_score(X, enriched_trades['gmm_cluster'])
agg_sil = silhouette_score(X, enriched_trades['agg_cluster'])
print(f"✅ GMM Silhouette: {gmm_sil:.3f}")
print(f"✅ Agglomerative Silhouette: {agg_sil:.3f}")

# Auto-label clusters
def label_cluster(idx):
    pts = X[enriched_trades['gmm_cluster'] == idx]
    if len(pts) == 0: return f'cluster_{idx}'
    center = dict(zip(feat_cols, pts.mean(axis=0)))
    if center.get('revenge_score', 0) > 0.5: return 'reactive_emotional'
    if center.get('size_deviation', 0) > 0.5 and center.get('trade_freq_7d', 0) > 0.3: return 'aggressive_high_activity'
    if center.get('size_deviation', 0) < -0.3 and center.get('trade_freq_7d', 0) < -0.3: return 'conservative_selective'
    return 'balanced_tactical'

cluster_names = {i: label_cluster(i) for i in range(4)}
enriched_trades['cluster_name'] = enriched_trades['gmm_cluster'].map(cluster_names)
print(f"   Cluster distribution: {enriched_trades['cluster_name'].value_counts().to_dict()}")


✅ GMM Silhouette: 0.131
✅ Agglomerative Silhouette: 0.168
   Cluster distribution: {'balanced_tactical': 40, 'reactive_emotional': 4}


### Cell 15: Dual HMM (User + Market)

In [22]:
try:
    # User HMM
    X_user = enriched_trades[['emotional_score','size_deviation']].fillna(0).to_numpy()
    user_hmm_model = hmm.GaussianHMM(n_components=3, covariance_type='diag', n_iter=200, random_state=42)
    user_hmm_model.fit(X_user)
    enriched_trades['hmm_state'] = user_hmm_model.predict(X_user)
    print(f"✅ User HMM — 3 behavioral states identified.")
    print("   Transition matrix:\n", user_hmm_model.transmat_.round(3))
except Exception as e:
    enriched_trades['hmm_state'] = 0
    print(f"⚠️  HMM: {e}")


✅ User HMM — 3 behavioral states identified.
   Transition matrix:
 [[0.325 0.472 0.203]
 [0.789 0.    0.211]
 [0.614 0.098 0.288]]


### Cell 16: Anomaly Detection (Isolation Forest + Autoencoder)

In [24]:
class Autoencoder(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(d,16), nn.ReLU(), nn.Linear(16,8))
        self.decoder = nn.Sequential(nn.Linear(8,16), nn.ReLU(), nn.Linear(16,d))
    def forward(self, x): return self.decoder(self.encoder(x))

X_t = torch.tensor(X.astype(np.float32)).to(device)

# Isolation Forest
iso = IsolationForest(contamination=0.1, random_state=42)
iso.fit(X)
iso_score = -iso.score_samples(X)

# Autoencoder
ae = Autoencoder(X.shape[1]).to(device)
opt_ae = optim.Adam(ae.parameters(), lr=0.01)
for epoch in range(60):
    opt_ae.zero_grad()
    loss = nn.MSELoss()(ae(X_t), X_t)
    loss.backward(); opt_ae.step()
ae.eval()
with torch.no_grad():
    recon = ae(X_t).cpu().numpy()
ae_error = np.mean((X - recon)**2, axis=1)
enriched_trades['ae_error'] = ae_error

# Combined anomaly score
iso_n = (iso_score - iso_score.min()) / (iso_score.max() - iso_score.min() + 1e-9)
ae_n  = (ae_error - ae_error.min()) / (ae_error.max() - ae_error.min() + 1e-9)
combined_score = 0.5*iso_n + 0.5*ae_n
thresh = np.quantile(combined_score, 0.9)
enriched_trades['anomaly_score']  = combined_score
enriched_trades['anomaly_flag']   = (combined_score > thresh).astype(int)
print(f"✅ Anomalies: {enriched_trades['anomaly_flag'].sum()} flagged ({enriched_trades['anomaly_flag'].mean():.1%})")


✅ Anomalies: 5 flagged (11.4%)


### Cell 17: Bayesian Network

In [25]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.estimators import MaximumLikelihoodEstimator, HillClimbSearch, BIC

try:
    bn_data = enriched_trades[['emotional_state','frequency_spike','is_loss']].astype(str)
    # Fixed causal structure: emotions + frequency → trade outcome
    model_bn = DiscreteBayesianNetwork([
        ('emotional_state', 'is_loss'),
        ('frequency_spike', 'is_loss')
    ])
    model_bn.fit(bn_data, estimator=MaximumLikelihoodEstimator)
    print("✅ Bayesian Network fitted: [emotional_state, frequency_spike] → [is_loss]")
    # Show CPD for is_loss
    from pgmpy.inference import VariableElimination
    infer = VariableElimination(model_bn)
    q = infer.query(['is_loss'], evidence={'emotional_state': 'euphoric'}, show_progress=False)
    print(f"   P(loss | euphoric): {dict(zip(q.state_names['is_loss'], q.values))}")
except Exception as e:
    print(f"⚠️  Bayesian Network: {e}")


✅ Bayesian Network fitted: [emotional_state, frequency_spike] → [is_loss]
   P(loss | euphoric): {'0': np.float64(0.0), '1': np.float64(1.0)}


### Cell 18: LSTM Sequential Pattern

In [26]:
class BehaviorLSTM(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, 64, num_layers=2, dropout=0.2, batch_first=True)
        self.fc   = nn.Linear(64, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

SEQ_LEN = 5
if len(X) > SEQ_LEN + 10:
    seqs  = np.array([X[i:i+SEQ_LEN] for i in range(len(X)-SEQ_LEN)])
    tgts  = enriched_trades['pnl'].to_numpy()[SEQ_LEN:]
    X_seq = torch.tensor(seqs.astype(np.float32)).to(device)
    y_seq = torch.tensor(tgts.astype(np.float32).reshape(-1,1)).to(device)
    lstm_model = BehaviorLSTM(X.shape[1]).to(device)
    opt_lstm = optim.Adam(lstm_model.parameters(), lr=0.001)
    for ep in range(40):
        opt_lstm.zero_grad()
        loss = nn.MSELoss()(lstm_model(X_seq), y_seq)
        loss.backward(); opt_lstm.step()
    lstm_model.eval()
    with torch.no_grad():
        preds = lstm_model(X_seq).cpu().numpy().flatten()
    lstm_errors = np.abs(tgts - preds)
    print(f"✅ LSTM trained. Mean prediction error: {lstm_errors.mean():.4f}")
else:
    print("⚠️  Not enough data for LSTM (need >15 trades). Skipping.")
    lstm_errors = np.zeros(max(len(X)-SEQ_LEN, 1))


✅ LSTM trained. Mean prediction error: 563.4520


### Cell 19: Risk Distribution & Copulas

In [27]:
returns  = (enriched_trades['pnl'] / enriched_trades['position_value'].replace(0, np.nan)).dropna()
var95    = float(np.quantile(returns, 0.05))
cvar95   = float(returns[returns <= var95].mean()) if (returns <= var95).any() else var95
ret_skew = float(skew(returns))
ret_kurt = float(kurtosis(returns))

try:
    cop_data = pd.DataFrame({
        'position_size': enriched_trades['position_value'].astype(float),
        'volatility':    enriched_trades.get('Volatility', pd.Series(np.zeros(len(enriched_trades)))).astype(float)
    }).replace([np.inf,-np.inf], np.nan).dropna()
    copula = GaussianMultivariate()
    copula.fit(cop_data)
    print("✅ Gaussian Copula fitted.")
except Exception as e:
    print(f"⚠️  Copula: {e}")

print(f"✅ Risk Profile — VaR95: {var95:.4f} | CVaR95: {cvar95:.4f}")
print(f"   Skewness: {ret_skew:.3f} | Kurtosis: {ret_kurt:.3f}")


✅ Gaussian Copula fitted.
✅ Risk Profile — VaR95: -0.0191 | CVaR95: -0.0231
   Skewness: -0.177 | Kurtosis: -1.620


### Cell 20: Granger Causality

In [28]:
granger_results = []
et = enriched_trades.copy()
et['regime_num'] = (et.get('market_regime', pd.Series(['normal']*len(et))) == 'trending_bullish').astype(float)
et['atr_val']    = et.get('ATR', pd.Series(np.zeros(len(et)))).astype(float).fillna(0)

def safe_granger(df, x_col, y_col, maxlag=3):
    pair = df[[y_col, x_col]].dropna()
    if len(pair) < maxlag + 10: return {'x': x_col, 'y': y_col, 'p_value': 1.0, 'story': 'insufficient data'}
    if pair[x_col].std() == 0 or pair[y_col].std() == 0:
        return {'x': x_col, 'y': y_col, 'p_value': 1.0, 'story': 'constant column — test skipped'}
    try:
        res  = grangercausalitytests(pair, maxlag=maxlag, verbose=False)
        best = min(res, key=lambda l: res[l][0]['ssr_ftest'][1])
        p    = float(res[best][0]['ssr_ftest'][1])
        return {'x': x_col, 'y': y_col, 'lag': best, 'p_value': p,
                'story': f'{x_col} → {y_col} (lag={best})' if p < 0.05 else f'No evidence {x_col} → {y_col}'}
    except Exception as ex:
        return {'x': x_col, 'y': y_col, 'p_value': 1.0, 'story': str(ex)}

tests = [('atr_val','size_deviation'), ('regime_num','emotional_score'), ('emotional_score','pnl')]
for x, y in tests:
    r = safe_granger(et, x, y)
    granger_results.append(r)
    sig = "✅" if r['p_value'] < 0.05 else "  "
    print(f"  {sig} p={r['p_value']:.4f} | {r['story']}")


     p=0.0853 | No evidence atr_val → size_deviation
     p=0.3242 | No evidence regime_num → emotional_score
     p=0.1151 | No evidence emotional_score → pnl


### Cell 21: Prospect Theory & Disposition Effect

In [29]:
def pt_value(x, alpha, lmbda):
    return np.where(x >= 0, np.power(x + 1e-9, alpha), -lmbda * np.power(np.abs(x) + 1e-9, alpha))

outcomes = returns.dropna().to_numpy()
x_n = np.linspace(-1, 1, len(outcomes))
y_n = np.tanh(outcomes / (np.std(outcomes) + 1e-9))
try:
    params, _ = curve_fit(pt_value, x_n, y_n, p0=[0.88, 2.25], maxfev=5000)
    alpha_pt, lambda_pt = float(params[0]), float(params[1])
except:
    alpha_pt, lambda_pt = 0.88, 2.25

pgr = (enriched_trades['pnl'] > 0).sum()
plr = (enriched_trades['pnl'] < 0).sum()
disposition_score = pgr / max(plr, 1)

print(f"✅ Prospect Theory — α: {alpha_pt:.3f} | λ (loss aversion): {lambda_pt:.3f}")
print(f"✅ Disposition Effect — PGR: {pgr} | PLR: {plr} | Score: {disposition_score:.2f}")
print(f"   {'🔴 Strong loss aversion' if lambda_pt > 2.5 else '🟡 Moderate' if lambda_pt > 1.5 else '🟢 Low'} loss aversion")
print(f"   {'🔴 Strong disposition effect' if disposition_score > 1.5 else '🟡 Moderate' if disposition_score > 1.0 else '🟢 Low'} disposition effect")


✅ Prospect Theory — α: 31.340 | λ (loss aversion): -0.708
✅ Disposition Effect — PGR: 25 | PLR: 19 | Score: 1.32
   🟢 Low loss aversion
   🟡 Moderate disposition effect


### Cell 22: GARCH Volatility Modelling

In [30]:
bvol_arr = enriched_trades['pnl'].astype(float).fillna(0).to_numpy()
try:
    gm = arch_model(bvol_arr, vol='Garch', p=1, q=1, dist='normal')
    gm_res = gm.fit(disp='off')
    enriched_trades['conditional_vol'] = np.asarray(gm_res.conditional_volatility, dtype=float)
    print(f"✅ GARCH fitted. Mean conditional vol: {enriched_trades['conditional_vol'].mean():.4f}")
except Exception as e:
    enriched_trades['conditional_vol'] = enriched_trades['pnl'].rolling(5).std().fillna(0)
    print(f"⚠️  GARCH fallback to rolling std: {e}")


✅ GARCH fitted. Mean conditional vol: 573.6484


### Cell 23: SHAP Explainability

In [33]:
try:
    # Align X with enriched_trades to ensure same length
    n = min(len(X), len(enriched_trades))
    X_shap = X[:n]
    y_shap = enriched_trades['gmm_cluster'].iloc[:n].values

    rf = RandomForestClassifier(n_estimators=50, random_state=42)
    rf.fit(X_shap, y_shap)
    explainer = shap.TreeExplainer(rf)
    shap_values = explainer.shap_values(X_shap)

    # Safely extract SHAP values for Cluster 0 regardless of SHAP version
    if isinstance(shap_values, list):
        class_0_shap = shap_values[0]  # Older SHAP: list of arrays
    else:
        class_0_shap = shap_values[:, :, 0]  # Newer SHAP: 3D array (samples, features, classes)

    feature_imp = pd.DataFrame({
        'feature': feat_cols,
        'shap_importance': np.abs(class_0_shap).mean(axis=0)
    }).sort_values('shap_importance', ascending=False)

    print("✅ SHAP computed. Top 5 behavioral drivers:")
    display(feature_imp.head())

except Exception as e:
    print(f"⚠️  SHAP: {e}")

✅ SHAP computed. Top 5 behavioral drivers:


,feature,shap_importance
12,ATR,0.101487
4,post_loss_hours,0.035258
9,pnl,0.033833
1,size_deviation,0.028121
8,holding_duration,0.017316


## Section 6 — Visualizations
### Cell 24: 1 — PnL & Emotional State Timeline

In [34]:
template = 'plotly_dark'
color_map = {'calm': '#00FF41', 'anxious': '#FFFF00', 'euphoric': '#FF0000'}

fig1 = px.scatter(
    enriched_trades, x='timestamp', y='pnl',
    color='emotional_state', color_discrete_map=color_map,
    size='position_value', hover_data=['symbol','cluster_name','hmm_state'],
    title='1. PnL & Emotional State Over Time', template=template
)
fig1.add_hline(y=0, line_color='gray', line_dash='dot')
fig1.update_layout(height=450)
fig1.show()


### Cell 25: 2 — HMM State Timeline

In [35]:
fig2 = make_subplots(rows=2, cols=1, shared_xaxes=True,
                     subplot_titles=['HMM Behavioral State', 'PnL'],
                     vertical_spacing=0.08)
fig2.add_trace(go.Scatter(
    x=enriched_trades['timestamp'], y=enriched_trades['hmm_state'],
    mode='lines+markers', line_shape='hv', name='HMM State',
    marker=dict(color=enriched_trades['hmm_state'],
                colorscale='Viridis', showscale=True, size=7)
), row=1, col=1)
fig2.add_trace(go.Bar(
    x=enriched_trades['timestamp'], y=enriched_trades['pnl'],
    name='PnL',
    marker_color=np.where(enriched_trades['pnl'] > 0, '#00FF41', '#FF0000')
), row=2, col=1)
fig2.update_layout(template=template, height=500,
                   title='2. HMM Behavioral State Sequence + PnL')
fig2.show()


### Cell 26: 3 — Behavioral Cluster Map

In [36]:
fig3 = px.scatter(
    enriched_trades, x='size_deviation', y='revenge_score',
    color='cluster_name', size='position_value',
    hover_data=['symbol','pnl','emotional_state'],
    title='3. Behavioral Cluster Map (Size Deviation vs Revenge Score)',
    template=template, color_discrete_sequence=px.colors.qualitative.Set2
)
fig3.update_layout(height=480)
fig3.show()


### Cell 27: 4 — Return Distribution & Risk

In [37]:
xs = np.linspace(float(returns.min()), float(returns.max()), 200)
try:
    kde = gaussian_kde(returns)
    ys  = kde(xs) * len(returns) * (xs[1]-xs[0])
except: ys = np.zeros_like(xs)

fig4 = go.Figure()
fig4.add_trace(go.Histogram(x=returns, nbinsx=35, name='Return Distribution',
    marker_color='rgba(52,152,219,0.5)'))
fig4.add_trace(go.Scatter(x=xs, y=ys, mode='lines', name='KDE',
    line=dict(color='#00FF41', width=2)))
fig4.add_vline(x=var95,  line_dash='dash', line_color='orange',
    annotation_text=f'VaR95={var95:.3f}')
fig4.add_vline(x=cvar95, line_dash='dash', line_color='red',
    annotation_text=f'CVaR95={cvar95:.3f}')
fig4.update_layout(template=template, height=420,
    title='4. Return Distribution with VaR / CVaR')
fig4.show()


### Cell 28: 5 — Behavioral Bias Radar

In [38]:
bias_scores = [
    min(disposition_score / 3.0, 1.0),
    min(float(enriched_trades['revenge_score'].mean()) * 2, 1.0),
    min(float(enriched_trades['size_deviation'].abs().mean()), 1.0),
    min(float(enriched_trades['early_exit'].mean()) * 2, 1.0),
    min(abs(lambda_pt - 1.0) / 3.0, 1.0),
]
bias_labels = ['Disposition Effect','Revenge Trading',
               'Overconfidence','Early Exit','Loss Aversion']

fig5 = go.Figure(go.Scatterpolar(
    r=bias_scores + [bias_scores[0]],
    theta=bias_labels + [bias_labels[0]],
    fill='toself', fillcolor='rgba(255,0,0,0.25)',
    line=dict(color='#FF0000', width=2), name='Bias Profile'
))
fig5.update_layout(
    polar=dict(radialaxis=dict(range=[0,1], tickvals=[0.25,0.5,0.75,1.0])),
    template=template, title='5. Behavioral Bias Scorecard', height=500
)
fig5.show()


### Cell 29: 6 — Anomaly Trade Map

In [39]:
fig6 = px.scatter(
    enriched_trades, x='timestamp', y='pnl',
    color=enriched_trades['anomaly_flag'].map({0:'normal',1:'anomalous'}),
    color_discrete_map={'normal':'#00FF41','anomalous':'#FF0000'},
    size='anomaly_score', hover_data=['symbol','emotional_state','cluster_name'],
    title='6. Anomaly Trade Detection', template=template
)
fig6.update_layout(height=430)
fig6.show()


### Cell 30: 7 — Win Rate by Market Regime

In [40]:
if 'market_regime' in enriched_trades.columns and enriched_trades['market_regime'].nunique() > 1:
    rdf = enriched_trades.groupby('market_regime')['pnl'].agg(
        win_rate=lambda x: (x>0).mean(),
        count='count', avg_pnl='mean'
    ).reset_index()
    fig7 = px.bar(rdf, x='market_regime', y='win_rate',
        color='avg_pnl', color_continuous_scale='RdYlGn',
        text=rdf['count'].apply(lambda n: f'n={n}'),
        title='7. Win Rate by Market Regime', template=template)
    fig7.update_layout(yaxis_tickformat='.0%', height=400)
    fig7.show()
else:
    print("Not enough regime diversity for this chart.")


### Cell 31: 8 — Prospect Theory Value Curve

In [41]:
x_r = np.linspace(-2, 2, 200)
y_r = np.where(x_r >= 0,
    np.power(np.maximum(x_r,0)+1e-9, alpha_pt),
    -lambda_pt * np.power(np.maximum(-x_r,0)+1e-9, alpha_pt))

fig8 = go.Figure()
fig8.add_trace(go.Scatter(x=x_r[x_r>=0], y=y_r[x_r>=0],
    name='Gains', line=dict(color='#00FF41', width=3)))
fig8.add_trace(go.Scatter(x=x_r[x_r<0],  y=y_r[x_r<0],
    name='Losses', line=dict(color='#FF0000', width=3)))
fig8.add_hline(y=0, line_color='gray'); fig8.add_vline(x=0, line_color='gray')
fig8.update_layout(template=template, height=420,
    title=f'8. Prospect Theory Value Function  (λ={lambda_pt:.2f}, α={alpha_pt:.2f})',
    xaxis_title='Outcome', yaxis_title='Subjective Value')
fig8.show()


### Cell 32: 9 — SHAP Feature Importance

In [42]:
try:
    pca = PCA(n_components=2)
    pca_coords = pca.fit_transform(X)
    pca_df = pd.DataFrame(pca_coords, columns=['PC1','PC2'])
    pca_df['cluster'] = enriched_trades['cluster_name'].values
    pca_df['pnl']     = enriched_trades['pnl'].values

    fig9 = px.scatter(pca_df, x='PC1', y='PC2', color='cluster',
        hover_data=['pnl'], size=np.abs(pca_df['pnl'])+0.1,
        title='9. PCA Cluster Projection (2D)',
        template=template, color_discrete_sequence=px.colors.qualitative.Pastel)
    fig9.update_layout(height=460)
    fig9.show()
except Exception as e: print(f"PCA chart error: {e}")


## Section 7 — Behavioral Report
### Cell 33: Generate Full Report

In [46]:
import json
import numpy as np

# 1. Define a custom encoder to automatically convert any NumPy types
class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        if isinstance(obj, np.floating):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super(NumpyEncoder, self).default(obj)

# Recalculate top-level variables
dominant_emotion  = enriched_trades['emotional_state'].value_counts().index[0]
dominant_cluster  = enriched_trades['cluster_name'].value_counts().index[0]
best_regime  = enriched_trades.groupby('market_regime')['pnl'].apply(lambda x: (x>0).mean()).idxmax() \
               if enriched_trades['market_regime'].nunique() > 1 else 'N/A'
worst_regime = enriched_trades.groupby('market_regime')['pnl'].apply(lambda x: (x>0).mean()).idxmin() \
               if enriched_trades['market_regime'].nunique() > 1 else 'N/A'

trading_style = (
    'disciplined_systematic' if float(enriched_trades['revenge_score'].mean()) < 0.2 else
    'reactive_emotional'     if float(enriched_trades['revenge_score'].mean()) > 0.35 else
    'overconfident_momentum' if lambda_pt > 2.5 else
    'mixed_adaptive'
)

# Build the report (using standard types where possible, but the encoder acts as a safety net)
report = {
    'session_id': str(session_id),
    'summary': {
        'total_trades':  int(len(enriched_trades)),
        'win_rate':      float((enriched_trades['pnl'] > 0).mean()),
        'total_pnl':     float(enriched_trades['pnl'].sum()),
        'avg_pnl':       float(enriched_trades['pnl'].mean()),
    },
    'behavioral_profile': {
        'trading_style':      str(trading_style),
        'dominant_cluster':   str(dominant_cluster),
        'dominant_emotion':   str(dominant_emotion),
        'gmm_silhouette':     float(gmm_sil),
    },
    'biases': {
        'loss_aversion_lambda':    float(lambda_pt),
        'risk_seeking_alpha':      float(alpha_pt),
        'disposition_score':       float(disposition_score),
        'revenge_trading_rate':    float(enriched_trades['revenge_score'].mean()),
        'early_exit_rate':         float(enriched_trades['early_exit'].mean()),
    },
    'risk_profile': {
        'var95': float(var95), 'cvar95': float(cvar95),
        'skewness': float(ret_skew), 'kurtosis': float(ret_kurt),
        'best_regime': str(best_regime), 'worst_regime': str(worst_regime),
    },
    'anomaly': {
        'anomaly_rate':   float(enriched_trades['anomaly_flag'].mean()),
        'anomaly_count':  int(enriched_trades['anomaly_flag'].sum()),
    },
    'causality': granger_results
}

# 2. Use the custom encoder (cls=NumpyEncoder) when dumping
with open(f'{base_path}/behavioral_report.json', 'w') as f:
    json.dump(report, f, indent=2, cls=NumpyEncoder)

def flag(val, high, med, reverse=False):
    if reverse: val = -val; high = -high; med = -med
    if val > high: return '🔴 HIGH'
    if val > med:  return '🟡 MODERATE'
    return '🟢 LOW'

print('=' * 60)
print('   BEHAVIORAL TRADING ANALYSIS REPORT')
print('=' * 60)
print(f"\n📊 Session Summary")
print(f"   Trades:    {report['summary']['total_trades']}")
print(f"   Win rate:  {report['summary']['win_rate']:.1%}")
print(f"   Total PnL: {report['summary']['total_pnl']:.2f}")
print(f"\n🧠 Behavioral Profile")
print(f"   Trading style:    {trading_style}")
print(f"   Dominant cluster: {dominant_cluster}")
print(f"   Dominant emotion: {dominant_emotion}")
print(f"\n⚠️  Bias Scorecard")
print(f"   Loss aversion (λ):  {lambda_pt:.3f} — {flag(lambda_pt, 2.5, 1.5)}")
print(f"   Disposition score:  {disposition_score:.3f} — {flag(disposition_score, 1.5, 1.0)}")
print(f"   Revenge rate:       {enriched_trades['revenge_score'].mean():.3f} — {flag(enriched_trades['revenge_score'].mean(), 0.35, 0.2)}")
print(f"   Early exit rate:    {enriched_trades['early_exit'].mean():.1%}")
print(f"\n📉 Risk Profile")
print(f"   VaR95:     {var95:.4f}")
print(f"   CVaR95:    {cvar95:.4f}")
print(f"   Best regime:  {best_regime}")
print(f"   Worst regime: {worst_regime}")
print(f"\n🔴 Anomalies: {report['anomaly']['anomaly_count']} trades ({report['anomaly']['anomaly_rate']:.1%})")
print(f"\n✅ Report saved to {base_path}/behavioral_report.json")

   BEHAVIORAL TRADING ANALYSIS REPORT

📊 Session Summary
   Trades:    44
   Win rate:  56.8%
   Total PnL: 8725.00

🧠 Behavioral Profile
   Trading style:    disciplined_systematic
   Dominant cluster: balanced_tactical
   Dominant emotion: calm

⚠️  Bias Scorecard
   Loss aversion (λ):  -0.708 — 🟢 LOW
   Disposition score:  1.316 — 🟡 MODERATE
   Revenge rate:       0.107 — 🟢 LOW
   Early exit rate:    11.4%

📉 Risk Profile
   VaR95:     -0.0191
   CVaR95:    -0.0231
   Best regime:  ranging
   Worst regime: trending_bearish

🔴 Anomalies: 5 trades (11.4%)

✅ Report saved to /content/behavioral_analysis/behavioral_report.json


## Section 8 — RAG Index & Groq Chatbot
### Cell 34: Build ChromaDB Index

In [47]:
chroma_client = chromadb.PersistentClient(path=f'{base_path}/chroma_db')
emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name='all-MiniLM-L6-v2')

session_coll = chroma_client.get_or_create_collection('session_data',  embedding_function=emb_fn)
static_coll  = chroma_client.get_or_create_collection('static_knowledge', embedding_function=emb_fn)

# Session chunks from analysis results
session_chunks = [
    f"Trading style is {trading_style}. Dominant behavioral cluster: {dominant_cluster}. Dominant emotional state: {dominant_emotion}.",
    f"Loss aversion lambda = {lambda_pt:.3f}. Disposition score = {disposition_score:.3f}. Revenge rate = {enriched_trades['revenge_score'].mean():.3f}.",
    f"VaR95 = {var95:.4f}. CVaR95 = {cvar95:.4f}. Skewness = {ret_skew:.3f}. Kurtosis = {ret_kurt:.3f}.",
    f"Best performing market regime: {best_regime}. Worst: {worst_regime}.",
    f"HMM identified 3 behavioral states. GMM silhouette: {gmm_sil:.3f}.",
    f"Anomaly detection flagged {enriched_trades['anomaly_flag'].sum()} trades ({enriched_trades['anomaly_flag'].mean():.1%}).",
    f"Win rate: {(enriched_trades['pnl']>0).mean():.1%}. Total PnL: {enriched_trades['pnl'].sum():.2f}.",
]
for g in granger_results:
    session_chunks.append(f"Causality: {g['story']} (p={g['p_value']:.4f})")

# Counterfactual chunks
cf_scenarios = {
    'half position size':        enriched_trades['pnl'] * 0.5,
    'skip trades after losses':  enriched_trades[enriched_trades['is_loss'].shift(1).fillna(0)==0]['pnl'],
    'only calm-state trades':    enriched_trades[enriched_trades['emotional_state']=='calm']['pnl'],
}
orig_pnl = enriched_trades['pnl'].sum()
for label, cf_pnl in cf_scenarios.items():
    delta = cf_pnl.sum() - orig_pnl
    session_chunks.append(
        f"Counterfactual — if trader used '{label}': PnL delta = {delta:.2f} "
        f"(original={orig_pnl:.2f}, counterfactual={cf_pnl.sum():.2f}).")

session_coll.upsert(documents=session_chunks, ids=[f'chunk_{i}' for i in range(len(session_chunks))])

# Static knowledge base
static_docs = [
    "Loss aversion (λ > 2) causes traders to hold losers too long and cut winners short. It's the most documented bias in behavioral finance.",
    "Disposition effect: tendency to realise gains quickly and defer losses. PGR/PLR > 1.5 indicates strong disposition bias.",
    "Revenge trading: impulsive larger trades placed within hours of a loss. Revenge traders show frequency spikes and size increases post-loss.",
    "Trending_bullish regime: momentum strategies work well. Use trailing stops. Avoid mean-reversion entries.",
    "High volatility regime: reduce position size to 0.25-0.5x base. Widen stops. Avoid large directional bets.",
    "VaR 95% is the max expected loss 95% of days. CVaR (Expected Shortfall) is the average loss when VaR is breached — a more tail-sensitive measure.",
    "Kelly criterion: optimal fraction = (bp - q) / b. Most traders use 0.25-0.5x Kelly to limit drawdowns.",
    "Overconfidence in trading manifests as excessive trade frequency, under-diversification, and ignoring stop losses.",
    "Anchoring bias: traders fixate on purchase price rather than current fair value, distorting exit decisions.",
]
static_coll.upsert(documents=static_docs, ids=[f'static_{i}' for i in range(len(static_docs))])

print(f"✅ ChromaDB indexed: {len(session_chunks)} session + {len(static_docs)} static chunks.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ ChromaDB indexed: 13 session + 9 static chunks.


### Cell 35: Configure Groq LLM

In [48]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage

if not GROQ_API_KEY:
    print("⚠️  No Groq API key. Add GROQ_API_KEY to Runtime > Secrets.")
else:
    llm = ChatGroq(model='llama-3.3-70b-versatile', temperature=0.1, api_key=GROQ_API_KEY)
    print("✅ Groq LLM ready (llama-3.3-70b-versatile — free tier).")

SYSTEM_PROMPT = f"""You are an expert Behavioral Trading Analyst. You have run a full analysis on this trader.

TRADER PROFILE:
- Style: {trading_style}
- Dominant cluster: {dominant_cluster}
- Loss aversion (λ): {lambda_pt:.3f}
- Disposition score: {disposition_score:.3f}
- Win rate: {(enriched_trades['pnl']>0).mean():.1%}
- VaR 95%: {var95:.4f}
- Best regime: {best_regime} | Worst: {worst_regime}

RULES:
1. Always ground answers in the retrieved context provided.
2. Tag your sources: [PROFILE], [RISK], [BIAS], [COUNTERFACTUAL], [KNOWLEDGE]
3. Be direct and specific. Avoid generic advice.
4. If the context doesn't cover the question, say so explicitly."""


✅ Groq LLM ready (llama-3.3-70b-versatile — free tier).


### Cell 36: Specialist Agent Functions

In [49]:
def classify_intent(msg):
    m = msg.lower()
    if any(k in m for k in ['bias','loss aversion','disposition','revenge','overconfid']): return 'bias_query'
    if any(k in m for k in ['risk','var','cvar','drawdown','volatility','tail']): return 'risk_query'
    if any(k in m for k in ['what if','counterfactual','if i had','scenario','alternative']): return 'counterfactual'
    if any(k in m for k in ['regime','market','stock','recommend','should i buy']): return 'market_query'
    if any(k in m for k in ['cluster','segment','group','type of trader']): return 'cluster_query'
    return 'general'

def behavior_agent(enriched_trades):
    top_cluster = enriched_trades['cluster_name'].value_counts().index[0]
    state = enriched_trades['emotional_state'].value_counts().index[0]
    return {'cluster': top_cluster, 'emotional_state': state,
            'revenge_rate': float(enriched_trades['revenge_score'].mean())}

def risk_agent(enriched_trades, var95, cvar95):
    return {'var95': var95, 'cvar95': cvar95,
            'worst_regime': worst_regime, 'anomaly_count': int(enriched_trades['anomaly_flag'].sum())}

def counterfactual_agent(enriched_trades, orig_pnl):
    results = {}
    for label, cf in cf_scenarios.items():
        results[label] = {'delta': round(cf.sum() - orig_pnl, 2), 'cf_pnl': round(cf.sum(), 2)}
    return results

def retrieve_context(query, n_session=4, n_static=2):
    try:
        s = session_coll.query(query_texts=[query], n_results=n_session)['documents'][0]
        k = static_coll.query( query_texts=[query], n_results=n_static)['documents'][0]
        return s + k
    except: return []

print("✅ Specialist agents ready.")


✅ Specialist agents ready.


### Cell 37: Main Chat Function

In [50]:
def chat(message):
    if not GROQ_API_KEY:
        return "⚠️ Groq API key not set. Add it to Runtime > Secrets > GROQ_API_KEY"

    intent = classify_intent(message)

    # Route to specialist agent for structured context
    extra_ctx = ''
    if intent == 'bias_query':
        result = behavior_agent(enriched_trades)
        extra_ctx = f"[PROFILE] Cluster={result['cluster']}, Emotion={result['emotional_state']}, RevengeRate={result['revenge_rate']:.3f}\n"
    elif intent == 'risk_query':
        result = risk_agent(enriched_trades, var95, cvar95)
        extra_ctx = f"[RISK] VaR95={result['var95']:.4f}, CVaR95={result['cvar95']:.4f}, Anomalies={result['anomaly_count']}\n"
    elif intent == 'counterfactual':
        result = counterfactual_agent(enriched_trades, enriched_trades['pnl'].sum())
        extra_ctx = '[COUNTERFACTUAL] ' + ' | '.join([f"{k}: delta={v['delta']:.2f}" for k,v in result.items()]) + '\n'

    # RAG retrieval
    docs = retrieve_context(message)
    rag_ctx = '\n'.join([f'- {d}' for d in docs])

    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=f"Context:\n{extra_ctx}{rag_ctx}\n\nQuestion: {message}")
    ]
    try:
        reply = llm.invoke(messages)
        return reply.content
    except Exception as e:
        return f"❌ Error: {e}"

# Quick test
print("🧪 Quick test...")
print(chat("What are my worst trading biases?"))


🧪 Quick test...
Based on your profile [PROFILE], your worst trading biases appear to be Anchoring bias [BIAS] and Overconfidence [BIAS]. The Anchoring bias is evident in your tendency to fixate on the purchase price rather than the current fair value, which can distort your exit decisions. Overconfidence is also a concern, as it may lead to excessive trade frequency, under-diversification, and ignoring stop losses. 

It's worth noting that your calm emotional state [PROFILE] and disciplined_systematic trading style [PROFILE] are generally positive attributes, but the presence of these biases can still impact your trading performance. The counterfactual analysis [COUNTERFACTUAL] suggests that filtering out non-calm state trades could improve your PnL, implying that emotional state may also play a role in your trading decisions. However, Anchoring bias and Overconfidence are the explicit biases mentioned in your profile.


### Cell 38: Interactive Chat UI

In [51]:
chat_history = []

QUICK_PROMPTS = [
    "What are my worst trading biases?",
    "What is my risk profile?",
    "What if I had halved my position size?",
    "Which market regime should I focus on?",
    "What does my cluster say about my trading style?",
    "Show me the counterfactual analysis.",
]

def send_message(b=None):
    msg = input_text.value.strip()
    if not msg: return
    input_text.value = ''
    chat_history.append(('user', msg))
    response = chat(msg)
    chat_history.append(('assistant', response))
    refresh_display()

def refresh_display():
    lines = []
    for role, text in chat_history[-12:]:
        escaped = text.replace('<','&lt;').replace('>','&gt;')
        if role == 'user':
            lines.append(
                f'<div style="text-align:right;margin:6px 0">'
                f'<span style="background:#0d6efd;color:white;padding:8px 14px;'
                f'border-radius:16px 16px 0 16px;display:inline-block;max-width:72%;'
                f'font-size:13px">{escaped}</span></div>')
        else:
            lines.append(
                f'<div style="text-align:left;margin:6px 0">'
                f'<span style="background:#1e1e1e;color:#ddd;padding:8px 14px;'
                f'border-radius:16px 16px 16px 0;display:inline-block;max-width:82%;'
                f'font-size:13px;white-space:pre-wrap">{escaped}</span></div>')
    chat_display.value = '\n'.join(lines)

input_text   = widgets.Text(placeholder='Ask about your trading behavior...',
                            layout=widgets.Layout(width='76%'))
send_btn     = widgets.Button(description='Send ➤', button_style='primary',
                              layout=widgets.Layout(width='12%'))
clear_btn    = widgets.Button(description='Clear', button_style='warning',
                              layout=widgets.Layout(width='10%'))
chat_display = widgets.HTML(
    value='<p style="color:#888;font-size:13px;padding:10px">Chat history will appear here...</p>',
    layout=widgets.Layout(height='420px', border='1px solid #333',
                          overflow_y='auto', padding='8px', background='#111'))

quick_buttons = [widgets.Button(description=p[:40], layout=widgets.Layout(width='48%', margin='2px'))
                 for p in QUICK_PROMPTS]

def make_quick(prompt):
    def handler(b):
        input_text.value = prompt
        send_message()
    return handler

for btn, prompt in zip(quick_buttons, QUICK_PROMPTS):
    btn.on_click(make_quick(prompt))

def clear_chat(b):
    chat_history.clear()
    chat_display.value = '<p style="color:#888;font-size:13px;padding:10px">Chat cleared.</p>'

send_btn.on_click(send_message)
clear_btn.on_click(clear_chat)
input_text.on_submit(send_message)

display(widgets.VBox([
    widgets.HTML("<h3 style='color:#0d6efd;margin:8px 0'>🤖 Behavioral Trading Analyst — Groq + RAG</h3>"),
    chat_display,
    widgets.HBox([input_text, send_btn, clear_btn]),
    widgets.HTML("<p style='color:#888;font-size:12px;margin:4px 0'>Quick prompts:</p>"),
    widgets.GridBox(quick_buttons, layout=widgets.Layout(grid_template_columns='repeat(2, 1fr)'))
]))


### Cell 39: Counterfactual Analysis Summary

In [52]:
print('=' * 55)
print('  COUNTERFACTUAL ANALYSIS')
print('=' * 55)

orig = enriched_trades['pnl'].sum()
orig_wr = (enriched_trades['pnl'] > 0).mean()

cf_table = []
scenarios_full = {
    'Half position size':               enriched_trades['pnl'] * 0.5,
    'Double position size':             enriched_trades['pnl'] * 2.0,
    'Skip 1 trade after every loss':    enriched_trades[enriched_trades['is_loss'].shift(1).fillna(0)==0]['pnl'],
    'Only calm emotional state trades': enriched_trades[enriched_trades['emotional_state']=='calm']['pnl'],
    'Only best cluster trades':         enriched_trades[enriched_trades['gmm_cluster']==enriched_trades['gmm_cluster'].value_counts().index[0]]['pnl'],
}
for name, cf_pnl in scenarios_full.items():
    delta  = cf_pnl.sum() - orig
    cf_wr  = (cf_pnl > 0).mean()
    cf_table.append({'Scenario': name, 'CF PnL': round(cf_pnl.sum(),2),
                     'Δ PnL': round(delta,2), 'CF Win Rate': f'{cf_wr:.1%}',
                     '': '✅ Better' if delta > 0 else '❌ Worse'})

cf_df = pd.DataFrame(cf_table)
print(f"Original — PnL: {orig:.2f} | Win Rate: {orig_wr:.1%}\n")
display(cf_df)

fig_cf = px.bar(cf_df, x='Scenario', y='Δ PnL',
    color=cf_df['Δ PnL'].apply(lambda x: 'Better' if x > 0 else 'Worse'),
    color_discrete_map={'Better':'#00FF41','Worse':'#FF0000'},
    title='Counterfactual PnL Delta vs Original', template='plotly_dark')
fig_cf.add_hline(y=0, line_color='white', line_dash='dot')
fig_cf.update_layout(height=380)
fig_cf.show()


  COUNTERFACTUAL ANALYSIS
Original — PnL: 8725.00 | Win Rate: 56.8%



,Scenario,CF PnL,Δ PnL,CF Win Rate,
0,Half position size,4362.5,-4362.5,56.8%,❌ Worse
1,Double position size,17450.0,8725.0,56.8%,✅ Better
2,Skip 1 trade after every loss,1525.0,-7200.0,42.3%,❌ Worse
3,Only calm emotional state trades,10895.0,2170.0,64.1%,✅ Better
4,Only best cluster trades,11500.0,2775.0,94.1%,✅ Better


### Cell 40: Export Results

In [53]:
from google.colab import files

enriched_trades.to_csv(f'{base_path}/enriched_trades.csv', index=False)
cf_df.to_csv(f'{base_path}/counterfactual_results.csv', index=False)
print("✅ Files saved.")

print("\n📥 Downloading...")
files.download(f'{base_path}/behavioral_report.json')
files.download(f'{base_path}/enriched_trades.csv')
files.download(f'{base_path}/counterfactual_results.csv')


✅ Files saved.

📥 Downloading...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## 📌 Quick Reference

| Section | Cells | What it does |
|---|---|---|
| 1 — Setup | 1–4 | Install packages, imports, SQLite schema, Groq API key |
| 2 — Ingestion | 5–8 | Upload CSV, auto-detect columns, FIFO reconstruction, emergency exclusion |
| 3 — Market Context | 9–10 | Fetch yfinance OHLCV, compute 15 indicators, classify 6 regimes, enrich trades |
| 4 — Features | 11–12 | Revenge score, emotional state, frequency spike, size deviation, RSI alignment |
| 5 — Models | 13–23 | GMM, Agglomerative, HMM, Autoencoder, Bayesian Network, LSTM, Risk/Copulas, Granger, Prospect Theory, GARCH, SHAP |
| 6 — Visualizations | 24–32 | 9 interactive Plotly charts |
| 7 — Report | 33 | Full behavioral report with bias scorecard |
| 8 — Chat | 34–40 | ChromaDB RAG, Groq LLM, specialist agents, interactive widget, counterfactual analysis, export |

**To use:** Add `GROQ_API_KEY` to Runtime > Secrets, then run top to bottom.